# 01 — Exploratory Data Analysis

Calls the versioned `src/` pipeline rather than duplicating logic. Ends with the hypotheses the modelling stage tests.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.utils.config import get_project_root

root = get_project_root()
df = pd.read_parquet(root / 'data/interim/cleaned_transactions.parquet')
df.shape

In [ ]:
summary = df.groupby('customer_id').agg(
    last_purchase=('invoice_date', 'max'),
    frequency=('invoice', 'nunique'),
    monetary=('line_total', 'sum'),
)
summary['recency_days'] = (df['invoice_date'].max() - summary['last_purchase']).dt.days
summary[['recency_days', 'frequency', 'monetary']].describe()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
summary['recency_days'].hist(ax=ax[0]); ax[0].set_title('Recency (days)')
summary['frequency'].clip(upper=40).hist(ax=ax[1]); ax[1].set_title('Frequency (orders)')
summary['monetary'].clip(upper=5000).hist(ax=ax[2]); ax[2].set_title('Monetary (GBP)')
plt.tight_layout()

## Hypotheses

1. Recency is the strongest single churn driver in this cohort.
2. Return rate is a leading indicator of churn (dissatisfaction), independent of recency.
3. Monetary value is heavy-tailed; a linear model on raw spend will be unstable, so a Tweedie/log1p target is warranted.
4. Discount sensitivity separates customers who respond to vouchers from those who do not.